In [5]:
import asyncio
from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    input_guardrail,
    output_guardrail
)
import os
from dotenv import load_dotenv

from IPython.display import display, Markdown
import nest_asyncio
import pandas as pd
from typing import Literal
from pydantic import BaseModel, Field
from src.Agent_OCR import load_from_json
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("mck_openai_api_key") or os.getenv("OPENAI_API_KEY", "")
os.environ["OPENAI_BASE_URL"] = os.getenv("mck_openai_base_url") or os.getenv("OPENAI_BASE_URL", "")

## Load Data

In [2]:
message =load_from_json("../Data/Intermediate/Client_Input_OCR/W2_XL_input_clean_2999.json")
print(len(message))

3893


## Setup Guardrails

In [37]:
class SpouseProfile(BaseModel):
    name : str
    "Name of the spouse"
    occupation: str
    "Occupation of the spouse"
    birthYear: int
    "Birth year of the spouse"

class UserProfile(BaseModel):
    Social_Security_Number: str
    '''Employee Social Security Number'''
    Name: str
    '''Full Name of the Employee'''
    Email: str
    '''Email Address of the Employee'''
    filingStatus: Literal['Single', "Married Filing Jointly", "Married Filing Separately", "Head of Household", "Qualifying Widow(er)"]
    '''Tax filling status of the Employee'''
    dependents: int
    '''Number of dependents of the Employee'''
    address: str
    '''Mailing Address of the Employee'''
    state: str
    '''State of residence of the Employee'''
    Occupation: str
    '''Occupation of the Employee'''
    spouse:SpouseProfile
    '''Spouse Information (if married)'''
    complete: Literal[True, False]
    ''' Provide a boolean value indicating whether all required fields are complete.'''
    complete_reasoning: str
    ''' Reasoning behind the completion of the user profile.'''
    missing_questions: list[str] = []
    ''' List of missing questions in the user profile.'''

class UserProfileOutput(BaseModel):
    reasoning: str
    ''' Reasoning for the guardrail output.'''
    is_complete: bool
    ''' Are all attributes in the user profile complete?'''
    missing_questions: list[str]
    ''' List of missing questions in the user profile.'''

PROMPT = '''
You are a helpful tax agent. You thoroughly read the document and provide the required information.
'''
UserProfileAgent = Agent(
        name="UserProfileAgent",
        instructions=PROMPT,
        model="o3",
        output_type=UserProfile,
        )

In [38]:
nest_asyncio.apply()
result = await Runner.run(
            UserProfileAgent, message
        )
result.final_output

UserProfile(Social_Security_Number='522-86-4190', Name='Daniel Robinson', Email='', filingStatus='Single', dependents=0, address='8432 Castro Well, West Anna, NJ 73837-7432', state='NJ', Occupation='', spouse=SpouseProfile(name='', occupation='', birthYear=0), complete=False, complete_reasoning='We were able to extract the taxpayer’s name, Social Security Number, and mailing address from the W-2 provided. No information was supplied about filing status, dependents, occupation, email, or spouse, which are all required to finalize the profile.', missing_questions=['What is your preferred email address?', 'What is your filing status for the 2010 tax year (Single, Married Filing Jointly, Married Filing Separately, Head of Household, Qualifying Widow(er))?', 'How many dependents do you claim?', 'What is your occupation?', 'Do you have a spouse to include on the return? If so, please provide your spouse’s full name, occupation, and year of birth.', 'Please confirm that New Jersey (NJ) is you

In [43]:
result = await Runner.run(
            UserProfileAgent, message
        )
while result.final_output.complete == False:
    r = {}
    for question in result.final_output.missing_questions:
        response = input(question)
        r[question] = response
    # Adjusted Profile
    adj_message = "# Additional User Information\n" + str(dict(r)) + "\n" + "# Original User Information\n" + str(message)
    result = await Runner.run(
            UserProfileAgent, adj_message
        )
dict(result.final_output)


{'Social_Security_Number': '522-86-4190',
 'Name': 'Daniel Robinson',
 'Email': 'Francisco_Reveriano@Mckinsey.com',
 'filingStatus': 'Single',
 'dependents': 0,
 'address': '8432 Castro Well, West Anna, NJ 73837-7432',
 'state': 'NJ',
 'Occupation': 'Consultant',
 'spouse': SpouseProfile(name='', occupation='', birthYear=0),
 'complete': True,
 'complete_reasoning': 'All required profile fields (name, SSN, email, filing status, dependents, occupation, address, and state) have been provided. Filing status is Single, so no spouse details are needed.',
 'missing_questions': []}